# Agent Evaluation Workbench

This notebook serves as the **Testing Workbench** for the Vacation Planner project. It allows for A/B testing of prompts, models, and tool configurations against standardized scenarios.

## 1. Setup

In [1]:
import os
import sys
import json
from typing import List, Dict
from dotenv import load_dotenv

# Load environment variables from the root .env file
load_dotenv(os.path.abspath("../.env"))

# Add the services/api directory to the path so we can import our agent logic
sys.path.append(os.path.abspath("../services/api"))

from agent.orchestrator import AgentOrchestrator
from agent.models import VacationPlan

print("Setup complete. Logic imported from services/api.")

Setup complete. Logic imported from services/api.


## 2. Define Scenarios

We define 'Golden Scenarios' to test the agent's adaptability and alignment.

In [2]:
SCENARIOS = [
    {
        "name": "The Anniversary Couple",
        "initial_input": "Hello! I want to plan a romantic trip for our 10th anniversary. I'm thinking Europe, but I'm not sure where.",
        "goal": "Identify 'Romantic' vibe, identify 'Europe' context, probe for budget/dates."
    },
    {
        "name": "The Budget Backpacker",
        "initial_input": "Help me find somewhere cheap in Southeast Asia. I have $1000 and two weeks. I love hiking.",
        "goal": "Identify 'Active' vibe, capture budget $1000, capture duration 14 days."
    }
]

## 3. A/B Testing Runner

Function to run a scenario against the current agent loop.

In [3]:
def run_test_case(scenario: Dict):
    print(f"--- Running Scenario: {scenario['name']} ---")
    orchestrator = AgentOrchestrator()
    plan = VacationPlan()
    history = [{"role": "user", "content": scenario['initial_input']}]
    
    response, updated_plan, new_msgs = orchestrator.run_turn(history, plan)
    
    print(f"Agent Response: {response}\n")
    print(f"Updated State: {updated_plan.model_dump_json(indent=2)}")
    
    # Simple check for goal alignment (manual for now)
    print(f"\nExpected Goal: {scenario['goal']}")

for s in SCENARIOS:
    run_test_case(s)

--- Running Scenario: The Anniversary Couple ---


Tool Call Detected: {"destination_candidates":["Europe"],"notes":"Romantic trip for 10th anniversary","status":"exploring"}


Agent Response: Europe is a wonderful continent for a romantic getaway. To help narrow down some options, can you tell me a bit more about your preferences? 

1. What is your approximate budget for this trip?
2. Are you thinking of traveling during a specific time of year or do you have flexible dates in mind?
3. What type of vibe are you looking for (e.g. city break, beach relaxation, cultural exploration, outdoor adventures)?

Updated State: {
  "destination_candidates": [
    "Europe"
  ],
  "budget_range": null,
  "travelers": 1,
  "dates": null,
  "requirements": [],
  "notes": "Romantic trip for 10th anniversary",
  "status": "exploring"
}

Expected Goal: Identify 'Romantic' vibe, identify 'Europe' context, probe for budget/dates.
--- Running Scenario: The Budget Backpacker ---


Tool Call Detected: {"budget_range":"$0-$1000","dates":"two weeks","requirements":["hiking"],"status":"narrowing"}


Agent Response: With a budget of $1000 and two weeks, you can explore some amazing hiking spots in Southeast Asia. Since you love hiking, I'll suggest a few destinations that fit your criteria.

Before I propose some options, I just want to confirm: Are you open to staying in budget-friendly accommodations, such as hostels or guesthouses, to make the most of your budget?

Also, are you interested in hiking in a specific type of terrain, such as mountains, forests, or coastal areas?

Updated State: {
  "destination_candidates": [],
  "budget_range": "$0-$1000",
  "travelers": 1,
  "dates": "two weeks",
  "requirements": [
    "hiking"
  ],
  "notes": "",
  "status": "narrowing"
}

Expected Goal: Identify 'Active' vibe, capture budget $1000, capture duration 14 days.


## 4. Next Steps

In Sprint 2, we will expand this notebook to support:
1. **Side-by-side prompt comparison**.
2. **Multi-turn simulations**.
3. **Automated Grade scores** (LLM-as-a-judge).